<h1 style="text-align: center;">Example of Medical Pipeline</h1>

## 🔍 Overview

In this notebook, we illustrate how to use **assetflow** to build a medical data processing pipeline based on data collected from patients over the course of a multi-month study.

## 📦 Imports

In [1]:
import polars as pl
from assetflow import Asset, asset

ImportError: cannot import name 'Asset' from 'assetflow' (unknown location)

## 🗂️ Data Description

The data have been simulated and are available in: <code>assetflow/data/simulated/generated</code>
    
They consist of **four tables**.

### `glucometers` Table

The `glucometers` table contains information about devices used to measure blood sugar levels (glucometers).  
It includes the following columns:

- **gm_id**: unique identifier of the glucometer  
- **model_name**: commercial name of the glucometer model  
- **manufacturer**: name of the company that manufactured the glucometer  
- **year**: year the glucometer model was released  
- **class**: class of glucometers (*Bluetooth*, *Optical*, *NFC*)

In [ ]:
glucometers = pl.read_csv(r"../data/simulated/generated/glucometers.csv")
glucometers

### `appointments` Table

The `appointments` table contains clinical measurements collected during patient medical visits over the course of the study.

It has a shape of **(1,647 rows × 8 columns)** and includes the following fields:

- **date**: date and time of the medical appointment  
- **patient_id**: unique identifier of the patient  
- **height**: patient height (in centimeters)  
- **weight**: patient weight (in kilograms)  
- **gm_id**: identifier of the glucometer used during the appointment  
- **glycemia**: measured blood glucose level  
- **bp_sys**: systolic blood pressure  
- **bp_dia**: diastolic blood pressure  

Each row corresponds to a single appointment for a given patient.


In [ ]:
appointments = pl.read_csv(r"../data/simulated/generated/appointments.csv")
appointments

### `patients` Table

The `patients` table contains demographic information about the patients participating in the study.

The data are split into two CSV files (`1.csv` and `2.csv`), partitioned by **sex**.

It includes the following columns:

- **patient_id**: unique identifier of the patient  
- **sex**: biological sex of the patient (encoded as integers)  
- **name**: patient name  
- **birthdate**: patient date of birth  

This table provides static patient-level information and can be joined with other tables using `patient_id`.


In [ ]:
patients = pl.read_csv(r"../data/simulated/generated/patients/*.csv")
patients

### `glycemia` Table

The `glycemia` table contains longitudinal blood glucose measurements recorded outside of scheduled appointments.

The data are partitioned by **year**, with one CSV file per year (e.g. `2015.csv`, `2016.csv`, …).

It includes the following columns:

- **patient_id**: unique identifier of the patient  
- **date**: timestamp of the glycemia measurement  
- **gm_id**: identifier of the glucometer used  
- **glycemia**: measured blood glucose level  

This table allows fine-grained temporal analysis of glycemia evolution over time.

In [ ]:
glycemia = pl.read_csv(r"../data/simulated/generated/glycemia/*.csv")
glycemia

## 📍 Creation of source Assets

Source assets represent objects that are only meant to be loaded. That is, we do not know how they are computed, but we want to use them later on.

In this pipeline, source assets represent the raw data of the study. Depending on whether the data are partitioned or not, the appropriate loaders are defined.

In [ ]:
glucometers = Asset(
    loader=CSVLoader(path=r"../data/simulated/generated/glucometers.csv")
)